In [1921]:
import pandas as pd
pd.set_option("mode.copy_on_write", True)
import numpy as np
from typing import no_type_check, Set, Sequence, Any,Optional,List,Callable,Dict,Union
import networkx as nx
import itertools
from collections import defaultdict
import sys
import os
sys.path.append(os.path.abspath('../../graph_rewrite'))
sys.path.append(os.path.abspath('../../spannerlib/spannerlib'))
from graph_rewrite.core import _create_graph, draw
from graph_rewrite.transform import rewrite, rewrite_iter

import time


from spannerlib.span import Span
from spannerlib.data_types import (
    Var, 
    FreeVar, 
    RelationDefinition, 
    Relation, 
    IEFunction,
    AGGFunction,
    IERelation, 
    Rule, 
    pretty
)
from spannerlib.ra import (
    _col_names,
    get_const,
    select,
    project,
    rename,
    union,
    intersection,
    difference,
    join,
    product,
    groupby,
    ie_map,
    merge_rows
)

from spannerlib.term_graph import graph_compose, merge_term_graphs_pair,rule_to_graph,add_relation,add_project_uniq_free_vars
from spannerlib.engine import Engine, IEFunction, AGGFunction


import logging
logger = logging.getLogger(__name__)

## Utils ##

In [1922]:
def schema_match(schema,expected,ignore_types=None):
    """checks if"""
    if len(schema) != len(expected):
        return False
    if ignore_types is None:
        ignore_types = []
    for x,y in zip(schema,expected):
        if x in ignore_types:
            continue
        if not issubclass(x,y):
            return False
    return True


def is_of_schema(relation,schema,ignore_types=None):
    """checks if a relation is of a given schema"""
    try:
        if len(relation) != len(schema):
            return False
        if ignore_types is None:
            ignore_types = []
        for x,y in zip(relation,schema):
            if type(x) in ignore_types:
                continue
            if not isinstance(x,y):
                return False
        return True
    except Exception as e:
        logger.error(f"Got Error when computing:\n"
                     f"is_of_scehma({relation},{schema})\n"
                     f"Error: {e}")
        raise e

def type_merge(type1,type2):
    if issubclass(type1,type2):
        return type1
    elif issubclass(type2,type1):
        return type2
    else:
        raise ValueError(f"Trying to merge types {type1},{type2}, types are incompatible")

def schema_merge(schema1,schema2):
    """merges two schemas, taking the stricter type between the two for each index"""
    if len(schema1) != len(schema2):
        raise ValueError(f"Trying to merge schemas {schema1},{schema2} schemas must be of the same length")
    
    new_schema = [type_merge(x,y) for x,y in zip(schema1,schema2)]
    return new_schema

In [1923]:
import re
STRING_PATTERN = re.compile(r"^[^\r\n]+$")

def isFloat(s):  
   n = '0123456789.' 
   return (all(x in n for x in s) and s.count('.') == 1)  
 
def isInt(s):  
   n = '0123456789'    
   return all(x in n for x in s) 

def _infer_relation_schema(row) -> Sequence[type]: # Inferred type list of the given relation
    """
    Guess the relation type based on the data.
    We support both the actual types (e.g. 'Span'), and their string representation ( e.g. `"[0,8)"`).

    **@raise** ValueError: if there is a cell inside `row` of an illegal type.
    """
    relation_types = []
    for cell in row:
        if not isinstance(cell, str):
            relation_types.append(type(cell))
        elif isInt(cell):
            relation_types.append(int)
        elif isFloat(cell):
            relation_types.append(float)
        elif cell in ['True', 'False']:
            relation_types.append(bool)
        else:
            relation_types.append(str)
        
    return relation_types

In [1924]:
class DB(dict):
    def __repr__(self):
        key_str=', '.join(self.keys())
        return f'DB({key_str})'

In [1925]:
def _col_names(length):
    # these names wont conflict with logical variables since they must always start with Uppercase letters
    return [f'col_{i}' for i in range(length)]

In [1926]:

# some select theta functions

class equalConstTheta():
    def __init__(self,*pos_val_tuples):
        self.pos_val_tuples = pos_val_tuples
    def __call__(self,df):
        masks = [df.iloc[:,pos]==val for pos,val in self.pos_val_tuples]
        return pd.concat(masks,axis=1).all(axis=1)
    def __str__(self):
        return f'''Theta({', '.join([f'col_{pos}={val}' for pos,val in self.pos_val_tuples])})'''
    def __repr__(self):
        return str(self)
    def __eq__(self,other):
        if not isinstance(other,equalConstTheta):
            return False
        return self.pos_val_tuples == other.pos_val_tuples

class equalColTheta():
    def __init__(self,*col_pos_tuples):
        self.col_pos_tuples = col_pos_tuples

    def __call__(self,df):
        masks = [df.iloc[:,pos1]==df.iloc[:,pos2] for pos1,pos2 in self.col_pos_tuples]
        return pd.concat(masks,axis=1).all(axis=1)    
    def __str__(self):
        return f'''Theta({', '.join([f'col_{pos1}=col_{pos2}' for pos1,pos2 in self.col_pos_tuples])})'''
    def __repr__(self):
        return str(self)
    def __eq__(self,other):
        if not isinstance(other,equalColTheta):
            return False
        return self.col_pos_tuples == other.col_pos_tuples

In [1927]:
def get_const(const_dict,**kwargs):
    return pd.DataFrame([const_dict])


def is_truthy(df):
    return df.shape==(1,0)

def is_falsy(df):
    return df.shape==(0,0)

In [1928]:
def select(df,theta,schema,**kwargs):
    if df is None or df.empty:
        return pd.DataFrame(columns=schema)
    if callable(theta):
        return df[theta(df)]
    else:
        raise ValueError(f"theta must be callable, got {theta}")

def project(df,schema,**kwargs):
    if df is None or df.empty:
        return pd.DataFrame(columns=schema)
    return df[schema]
    
def rename(df,schema,**kwargs):
    if df is None or df.empty:
        return pd.DataFrame(columns=schema)
    
    df=df.copy()
    df.columns = schema
    return df

def intersection(df1,df2,schema,**kwargs):
    if df1 is None or df2 is None or df1.empty or df2.empty:
        return pd.DataFrame(columns=schema)
    return pd.merge(df1,df2,how='inner',on=list(df1.columns))

def difference(df1,df2,schema,**kwargs):
    if df1 is None or df2 is None or df1.empty or df2.empty:
        return pd.DataFrame(columns=schema)
    return pd.concat([df1,df2]).drop_duplicates(keep=False)


def product(df1,df2,schema,**kwargs):
    if df1 is None or df2 is None or df1.empty or df2.empty:
        return pd.DataFrame(columns=schema)
    return pd.merge(df1,df2,how='cross')

def join(df1,df2,schema,**kwargs):
    if df1 is None or df2 is None or is_falsy(df1) or is_falsy(df2):
        return pd.DataFrame(columns=schema)

    # if one of the dataframes is truthy, return the other
    # this solves the problem of joining with a constant
    if is_truthy(df1):
        return df2
    if is_truthy(df2):
        return df1

    cols1 = set(df1.columns)
    cols2 = set(df2.columns)
    on = cols1 & cols2
    # get only logical variables
    # on = [ col for col in on if isinstance(col,str) and col[0].isupper()]
    on = list(on)
    if len(on)==0:
        return pd.merge(df1,df2,how='cross')
    else:
        return pd.merge(df1,df2,how='inner',on=on)
    
def merge_rows(*dfs):
    return pd.DataFrame(
        set.union(*[set(df.itertuples(index=False,name=None)) for df in dfs])
    )


def union(*dfs,schema,**kwargs):
    # use numpy arrays to ignore column names
    non_empty_dfs = []
    for df in dfs:
        if df is not None and not df.empty:
            non_empty_dfs.append(df)
    if len(non_empty_dfs)==0:
        return pd.DataFrame(columns=schema)
    else:
        return rename(merge_rows(*non_empty_dfs),schema)
        # This line didnt work since drop duplicates doesnt work correctly on non primitive classes such as Spans
        # return pd.DataFrame(np.concatenate(non_empty_dfs,axis=0),columns=schema).drop_duplicates(ignore_index=True)
        
def groupby(df,schema,agg,**kwargs):
    if df is None or df.empty:
        return pd.DataFrame(columns=schema)
    
    # rename columns to numbers so that we can aggregate the same free var to multiple places
    uniq_cols_df = rename(df,schema=[i for i in range(len(schema))])

    groupby_cols = [i for i,agg_func in enumerate(agg) if agg_func is None]
    agg_by_cols = {i:agg_func for i,agg_func in enumerate(agg) if agg_func is not None}
    # a real groupby
    if len(groupby_cols)>0:
        return rename(
            project(
                uniq_cols_df.groupby(groupby_cols).agg(agg_by_cols).reset_index(),
                schema = uniq_cols_df.columns
                ),
            schema)
    # no group by vars, so aggs contain all columns and schema simply orders them
    else:

        # this conversion magic is caused by an inconsistency between series and dataframes aggs,
        # to enable using both function and str aliases we
        # we take each column, convert to a frame
        # aggregate it and then squeeze it to a series (which has a single value)
        # then feed that to the dataframe constructor
        return rename(
            pd.DataFrame({
                col:[uniq_cols_df[col].to_frame().agg(agg_by_cols[col]).squeeze()] for col in range(len(agg_by_cols))
            }),
            schema)

In [1929]:
def coerce_tuple_like(name,func,input,output):
    if isinstance(output,(tuple,list)):
        return output
    else:
        logger.debug(f"IEFunction {name} with underlying function {func}\n"
                        f"returned a value that is not a tuple/list\n"
                        f"for input {input} -> {output}\n"
                        f"coercing to tuple")
        return (output,)

def assert_ie_schema(name,func,value,expected_schema,arity,input_or_output='input'):
    if callable(expected_schema):
        expected_schema = expected_schema(arity)
    if not is_of_schema(value,expected_schema):
        raise ValueError(
            f"IEFunction {name} with underlying function {func}\n"
            f"received an {input_or_output} value {value}(schema={pretty(_infer_relation_schema(value))})\n"
            f"but expected {pretty(expected_schema)}")

def assert_iterable(name,func,input,output):
    try:
        out_iter = iter(output)
    except TypeError:
        raise ValueError(f"IEFunction {name} with underlying function {func}\n"
                f"returned a value that is not an iterable\n"
                f"for input {input} -> {output}")

def map_iter(df,name,func,in_schema,out_schema,in_arity,out_arity,**kwargs):
    """helper function returns an iterator that applies a function to each row of a dataframe
    """
    for _,in_row in df.iterrows():
        in_row = list(in_row)
        assert_ie_schema(name,func,in_row,in_schema,in_arity,input_or_output='input')
        output = func(*in_row)
        assert_iterable(name,func,in_row,output)
        for out_row in output:
            out_row = coerce_tuple_like(name,func,in_row,out_row)
            out_row = list(out_row)
            assert_ie_schema(name,func,out_row,out_schema,out_arity,input_or_output='output')
            yield in_row + out_row

def ie_map(df,name,func,in_schema,out_schema,in_arity,out_arity,**kwargs):
    """given an indexed dataframe, apply an ie function to each row and return the output 
    such that each output relation is indexed by the same index as the input relation that generated it
    """
    if df is None or df.empty:
        return pd.DataFrame(columns=_col_names(in_arity+out_arity))
    output_iter = map_iter(df,name,func,in_schema,out_schema,in_arity,out_arity)
    total_arity = in_arity + out_arity
    return pd.DataFrame(output_iter,columns=_col_names(total_arity))

In [1930]:

def get_rel(rel,db,**kwargs):
    # helper function to get the relation from the db for external relations
    return db[rel]

op_to_func = {
    'union':union,
    'intersection':intersection,
    'difference':difference,
    'select':select,
    'project':project,
    'rename':rename,
    'join':join,
    'ie_map':ie_map,
    'get_rel':get_rel,
    'get_const':get_const,
    'product':product,
    'groupby':groupby
}

In [1931]:
def _in_cycle(g):
    return list(set(
        itertools.chain.from_iterable(nx.cycles.simple_cycles(g))
    ))

def _depends_on_cycle(g):
    in_cycle_nodes = _in_cycle(g)
    depends_on_cycle = {
        node for node in g.nodes if node in in_cycle_nodes or 
        len(set(nx.descendants(g,node)).intersection(in_cycle_nodes))>0
    }
    return depends_on_cycle

In [1932]:
def profile_wrapper(op_func, profile_data, children_results, u_data):
    start = time.time()
    res = op_func(*children_results, **u_data)
    end = time.time()
    profile_data[op_func.__name__]["count"] += 1
    profile_data[op_func.__name__]["total_time"] += end - start
    return res, end - start

In [1933]:
def _collect_children_and_run(G,u,results,profile_data,stack,log=False):
    children = list(G.successors(u))
    u_data = G.nodes[u]

    children_results = [results[v][-1] for v in children]
    op_func = op_to_func[u_data['op']]

    if log:
        logger.debug(f"computing node {u} with children {children} and data {u_data} , stack = {stack}")
        logger.debug(f"children results are {children_results}")
        logger.debug(f"children_data is {[G.nodes[v] for v in children]}")
    try:
        res, op_time = profile_wrapper(op_func, profile_data, children_results, u_data)
    except Exception as e:
        raise Exception(f'During excution of node {u} with args {children_results} and kwargs {u_data}'
                        f' got error {e}'
        )
    if log:
        logger.debug(f"result of node {u} is {res}")
    results[u].append(res)
    G.nodes[u]['op_time'] = op_time
    return res


In [1934]:
def compute_acyclic_node(G,u,results,profile_data,stack=None):
    res = _collect_children_and_run(G,u,results,profile_data,[])
    logger.debug(f"setting {u} to final since it is acyclic\n")
    G.nodes[u]['final'] = True
    return res

def compute_recursive_node(G,u,results,profile_data,stack=None):

    if stack is None:
        stack = []

    children = list(G.successors(u))
    u_data = G.nodes[u]
    op_func = op_to_func[u_data['op']]

    if u_data.get('final',False):
        return results[u][-1]    


    logger.debug(f"computing node {u} with stack {stack}")


    went_in_a_cycle = u in stack
    if went_in_a_cycle:
        logger.debug(f"went in a cycle at {u}, computing op with empty children if necessary\n")
        # for each child that doesnt have data, put an empty df instead of it
        res = _collect_children_and_run(G,u,results,profile_data,stack,log=True)
        return res

    # if we are here we are in a cycle but didnt return to an old position yet
    # then we compute all our children first
    for v in children:
        stack.append(u)
        compute_recursive_node(G,v,results,profile_data,stack)
        stack.pop()

    # compute and mark as final if reached fixed point
    res = _collect_children_and_run(G,u,results,profile_data,stack,log=True)

    all_children_final = all(G.nodes[v].get('final',False) for v in children)
    fixed_point_reached = len(results[u])>1 and results[u][-1].equals(results[u][-2])

    if all_children_final:
        logger.debug(f"setting {u} to final since all children are final\n")
        G.nodes[u]['final'] = True
    elif fixed_point_reached:
        logger.debug(f"setting {u} to final since fixed point has been achieved\n")
        # if u==9:
        #     logger.debug(f"graph nodes are{g.nodes(data=True)}")
        G.nodes[u]['final'] = True
    else:
        logger.debug(f"{u} not final yet so we will need to run another iteration\n")

    return res



def compute_node(G,root,ret_inter=False):

    # makes sure there is always a last value in the list for each key
    # which is None
    list_with_none_factory = lambda : [None]
    results_dict = defaultdict(list_with_none_factory)
    profile_data = defaultdict(lambda: {"count": 0, "total_time": 0.0})
    start_time = time.time()
    depends_on_cycle = _depends_on_cycle(G)
    not_depends_on_cycle = [u for u in G.nodes if u not in depends_on_cycle]

    # compute non cyclic nodes in postorder
    non_cycle_topological_sort = list(nx.topological_sort(nx
                                                          .DiGraph(nx.subgraph(G,not_depends_on_cycle))))
    for u in non_cycle_topological_sort[::-1]:
        compute_acyclic_node(G,u,results_dict,profile_data)

    logger.debug(f"the following nodes were computed non cyclically {non_cycle_topological_sort}")
    # now that all initial conditions for recursions are set
    # run the compute_recursive_node on u
    logger.debug(f"running compute_recursive_node on {root}")

    while True:
        res = compute_recursive_node(G,root,results_dict,profile_data)
        if G.nodes[root].get('final',False):
            break
    end_time = time.time()
    profile_data['total_time'] = end_time - start_time
    if ret_inter:
        return res,profile_data, results_dict
    else:
        return res, profile_data


In [1935]:
graph  = nx.DiGraph()
graph.add_nodes_from([
    0,1,2,3,
])
graph.add_edges_from(
    [(0,1),(0,2),(1,3),(2,3),(3,4)]
)
draw(graph)
edges_df = pd.DataFrame(list(graph.edges),columns=['S','T'])
edges_df
db = DB({
    'edges':edges_df
})

In [1936]:
g = nx.DiGraph()
g.add_nodes_from([
    ('edges',{'rel':'edges','op':'get_rel','db':db}),
    (1,{'op':'rename','schema':['S','T']}),
    (2,{'op':'rename','schema':['S','X']}),
    (3,{'op':'rename','schema':['X','T']}),
    (4,{'op':'join','schema':['S','X','T']}),
    (5,{'op':'project','schema':['S','T']}),
    ('reachable',{'op':'union','schema':[0,1]}),
    (6,{'op':'rename','schema':['S','T']})]
)
g.add_edges_from([
    (1,'edges'),
    (2,'edges'),
    (4,2),
    (4,3),
    (5,4),
    ('reachable',5),
    ('reachable',1),
    (3,'reachable'),
    (6,'reachable')
])
draw(g)

In [1937]:
res, profile_data, ret_inter = compute_node(g,6,ret_inter=True)
profile_df = pd.DataFrame(profile_data).T
profile_df

,count,total_time
get_rel,1.000000,0.000002
rename,8.000000,0.002610
union,6.000000,0.002380
join,3.000000,0.008362
project,3.000000,0.002990
total_time,0.049629,0.049629


In [1938]:
def func(str):
    yield (len(str),)
    
G = nx.DiGraph()

str_list = ["a" * n for n in range(20000)]

db = DB({
    "string": pd.DataFrame({"col_0": str_list}),
    "string_length": pd.DataFrame(columns=["col_0", "col_1"])  
})


nodes = {
    "string": {
        "op": "get_rel",
        "rel": "string",
        "rule_id": "{0, 'fact'}",
        "schema": ["col_0"],
        "db": db
    },
    "string_length": {
        "op": "union",
        "rel": "string_length",
        "rule_id": "{0, 'fact'}",
        "schema": ["col_0", "col_1"]
    },
    "0": {
        "op": "rename",
        "schema": ["Str"],
        "rule_id": "{0}"
    },
    "1": {
        "op": "project",
        "schema": ["Str"],
        "rule_id": "{0}"
    },
    "2": {
        "op": "project",
        "schema": ["Str"],
        "rule_id": "{0}"
    },
    "3": {
        "op": "ie_map",
        "func": func,
        "in_arity": 1,
        "out_arity": 1,
        "schema": ["col_0", "col_1"],
        "rule_id": "{0}",
        "name": "Length",
        "in_schema": [str],
        "out_schema": [int]
    },
    "4": {
        "op": "rename",
        "schema": ["Str", "Len"],
        "rule_id": "{0}"
    },
    "5": {
        "op": "rename",
        "schema": ["Str", "Len"],
        "rule_id": "{0}"
    },
    "6": {
        "op": "project",
        "schema": ["Str", "Len"],
        "rule_id": "{0}"
    },
    "7": {
        "op": "join",
        "schema": ["Str", "Len"],
        "rule_id": "{0}"
    },
    "8": {
        "op": "project",
        "schema": ["Str", "Len"],
        "rel": "_string_length_0",
        "rule_id": "{0}"
    },
    "9": {
        "op": "rename",
        "schema": ["Str", "Len"]
    },
    "10": {
        "op": "project",
        "schema": ["Str", "Len"]
    }
}

# Add nodes to the graph with their attributes
for node, attrs in nodes.items():
    G.add_node(node, **attrs)

# Define edges as per the Mermaid diagram structure
edges = [
    ("0", "string"),
    ("1", "0"),
    ("2", "1"),
    ("3", "2"),
    ("4", "3"),
    ("5", "4"),
    ("6", "5"),
    ("7", "6"),
    ("7", "1"),
    ("string_length", "8"),
    ("8", "7"),
    ("9", "string_length"),
    ("10", "9")
]

# Add edges to the graph
G.add_edges_from(edges)
draw(G)

In [1939]:
#db["string"]


In [1940]:
#res, profile_data = compute_node(G, "10")
# print res ordered by length
#print(res.sort_values(by="Len"))
#profile_df = pd.DataFrame(profile_data).T
#profile_df


# Query I - tax year 

In [1941]:
e = Engine()
e.set_relation(RelationDefinition(name='sms_rel', scheme=[str, int, bool, str, str, bool])) # sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam
    
def refund_for_year(sms_body):
    # search for the word refund and then the year
    match = re.search(r'refund for\s*([1-2][0-9]{3})', sms_body)
    if match:
        return [(str(match.group()),)]
    else:
        return []

def year_open_for_refund(sms_body):
    # search for a year, then somewhere the words 'open for', then somewhere the word 'refund'
    match = re.search(r'([1-2][0-9]{3})[a-zA-z ]*open for[a-zA-Z ]*refund', sms_body)
    if match:
        return [(str(match.group()),)]
    else:
        return []
    
e.set_ie_function(IEFunction(name='refund_for_year',func=refund_for_year,in_schema=[str],out_schema=[str]))
e.set_ie_function(IEFunction(name='year_open_for_refund',func=year_open_for_refund,in_schema=[str],out_schema=[str]))

refund_for_year_rule = Rule(
    head=Relation(name='sms_tax_year', terms=[FreeVar(name='sms_id'),FreeVar(name='sender_id'),FreeVar(name='is_sender_in_contacts'),FreeVar(name='sms_body'),
                                            FreeVar(name='sms_date'),FreeVar(name='is_spam')]),
    body=[
        Relation(name='sms_rel', terms=[FreeVar(name='sms_id'),FreeVar(name='sender_id'),FreeVar(name='is_sender_in_contacts'),FreeVar(name='sms_body'),
                                        FreeVar(name='sms_date'),FreeVar(name='is_spam')]),
        IERelation(name='refund_for_year', in_terms=[FreeVar(name='sms_body')], out_terms=[FreeVar(name='regex_result')]),
    ])

year_open_for_refund_rule = Rule(
    head=Relation(name='sms_tax_year', terms=[FreeVar(name='sms_id'),FreeVar(name='sender_id'),FreeVar(name='is_sender_in_contacts'),FreeVar(name='sms_body'),
                                            FreeVar(name='sms_date'),FreeVar(name='is_spam')]),
    body=[
        Relation(name='sms_rel', terms=[FreeVar(name='sms_id'),FreeVar(name='sender_id'),FreeVar(name='is_sender_in_contacts'),FreeVar(name='sms_body'),
                                        FreeVar(name='sms_date'),FreeVar(name='is_spam')]),
        IERelation(name='year_open_for_refund', in_terms=[FreeVar(name='sms_body')], out_terms=[FreeVar(name='regex_result')]),
    ])

e.add_rule(refund_for_year_rule,RelationDefinition(name='sms_tax_year', scheme=[str,int,bool,str,str,bool]))
e.add_rule(year_open_for_refund_rule,RelationDefinition(name='sms_tax_year', scheme=[str,int,bool,str,str,bool]))

sms_rel = pd.read_csv('sms_rel.csv')
e.add_facts('sms_rel',sms_rel)


query_graph, root_node = e.plan_query(Relation(name='sms_tax_year',terms=[FreeVar(name='sms_id'), FreeVar(name='sender_id'), FreeVar(name='is_sender_in_contacts'), FreeVar(name='sms_body'),
                                            FreeVar(name='sms_date'), FreeVar(name='is_spam')]))
res, profile_data = compute_node(query_graph, root_node)
draw(query_graph)

profile_df = pd.DataFrame(profile_data).T
print(profile_df)
res

               count  total_time
get_rel     1.000000    0.000005
rename      7.000000    0.004285
project     9.000000    0.016695
ie_map      2.000000    0.087309
join        2.000000    0.015348
union       1.000000    0.001646
total_time  0.129521    0.129521


,sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam
0,S789012A345677,073-3489675,False,"Hi Stav Compi, Information shows you have a ta...","28/11/2023, 13:55",True
1,S678901A234565,033-030413,False,This is the last chance to withdraw. If you ha...,"10/09/2024, 10:34",True
2,S567890A123457,FreeMoney,False,Nadav CS the tax refund for 2016 is about to e...,"31/10/2022, 13:14",True
3,S234567A890127,072-3491746,False,Check if you qualify for a tax refund for 2023...,"07/02/2024, 08:25",True
4,S345678A901233,033-030571,False,"Hi Nadav army, The year 2024 has arrived, and ...","03/07/2024, 13:40",True
5,S789012A345676,Missim,False,You might also receive such a message! Hello S...,"06/08/2024, 15:52",True
6,S123456A789010,TaxReturn,False,"Nadav RD 15, According to our records, you are...","05/01/2023, 20:27",True
7,S456789A012343,TaxReturn,False,"Stav Aviram - Army, According to our records, ...","22/01/2023, 11:32",True
8,S789012A345678,050-6105563,False,"Based on our system, you might be eligible for...","02/05/2023, 20:17",True
9,S352109A847627,FreeTax,False,Find out how much your tax refund for 2020 is ...,"06/01/2024, 13:25",True


# "Naive" optimization - removing redundant/unnecessary nodes

In [1942]:
# We start by removing all unnecesary project nodes - all projects here that are not childs of an ie_map node are redundant
def ie_map_x(match):
    if 'ie_map' in match['x']['op']:
        return False
    return True
            
for match in rewrite_iter(query_graph, lhs='project_node[op="project", schema];x[op]->project_node->y', p='x[op],y', rhs='x[op]->y',condition=lambda match:
    ie_map_x(match), is_recursive=True):
    pass
draw(query_graph)

In [1943]:
# There are 2 cases of calling rename twice in a row, we can merge them into a single rename
def schema(match, node):
    return match[node]['schema']
rewrite(query_graph, lhs='y[op="rename",schema]->z[schema];x->y', p='x,z[schema]', rhs='x->z[schema={{y_schema}}]',
        render_rhs={'y_schema': lambda match: schema(match, 'y')}, is_recursive=True)
draw(query_graph)

In [1944]:
# change the union to be before the join, remove the project root which is unnecessary, and make the root project node edge to the new join node
for match in rewrite_iter(query_graph,
        lhs='union_node[op="union",schema]->join_node[op="join",schema]->ie_node[func,schema],join_node->sms_rel',
        p='union_node[op],ie_node[func,schema],sms_rel',
        rhs='union_node[op,schema={{ie_node_schema}}]->ie_node[func,schema], sms_rel', 
        render_rhs={'ie_node_schema': lambda match: schema(match, 'ie_node'), 'join_node_schema': lambda match: schema(match, 'join_node')}):
        join_schema = match['join_node']['schema']
rewrite(query_graph, lhs='project_node[op="project"]->union_node[op="union"], sms_rel[op="get_rel",schema]', 
        p='project_node[op],sms_rel[op,schema],union_node[op]',
        rhs='project_node[op]->join_node[op="join",schema={{join_schema}}]->union_node[op], join_node->sms_rel[op,schema]',
        render_rhs={'join_schema': lambda match: join_schema})

draw(query_graph)

In [1945]:
# set sms_rel node db schema to ['sms_id', 'sender_id', 'is_sender_in_contacts', 'sms_body', 'sms_date', 'is_spam']
query_graph.nodes['sms_rel']['db'] = DB({'sms_rel': sms_rel, 'sms_tax_year': pd.DataFrame()})
query_graph.nodes['sms_rel']['db']['sms_rel']
print("Profile data before optimization")
print(profile_df)
res, profile_data = compute_node(query_graph, 19)
print()
print("Profile data after optimization")
profile_df = pd.DataFrame(profile_data).T
print(profile_df)
res


Profile data before optimization
               count  total_time
get_rel     1.000000    0.000005
rename      7.000000    0.004285
project     9.000000    0.016695
ie_map      2.000000    0.087309
join        2.000000    0.015348
union       1.000000    0.001646
total_time  0.129521    0.129521

Profile data after optimization
               count  total_time
get_rel     1.000000    0.000005
project     3.000000    0.005123
ie_map      2.000000    0.077512
union       1.000000    0.000985
join        1.000000    0.003135
total_time  0.088316    0.088316


,sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam
0,S123456A789013,RefundTax,False,Check if you are eligible for a tax refund for...,"29/11/2021, 18:12",True
1,S234567A890122,033-030571,False,"Hi Nadav army, The year 2024 has arrived, and ...","13/06/2024, 10:14",True
2,S223456A787614,072-3491746,False,Your tax refund for 2018 may be higher than yo...,"05/01/2024, 16:00",True
3,S345678A901233,033-030571,False,"Hi Nadav army, The year 2024 has arrived, and ...","03/07/2024, 13:40",True
4,S789012A345677,073-3489675,False,"Hi Stav Compi, Information shows you have a ta...","28/11/2023, 13:55",True
5,S890123A456787,TaxBack,False,We noticed that you might be eligible for a ta...,"28/08/2024, 10:07",True
6,S901234A567898,Missim,False,You might also receive such a message! Hello S...,"22/08/2024, 14:06",True
7,S234567A890124,MyReTax,False,Check your eligibility for a tax refund for 20...,"21/05/2022, 20:39",True
8,S789012A345678,050-6105563,False,"Based on our system, you might be eligible for...","02/05/2023, 20:17",True
9,S352109A847627,FreeTax,False,Find out how much your tax refund for 2020 is ...,"06/01/2024, 13:25",True


# Query II - tax link

In [1946]:
#e = Engine()
#e.set_relation(RelationDefinition(name='sms_rel', scheme=[str, int, bool, str, str, bool])) # sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam
#e.add_facts('sms_rel',sms_rel)

def find_link_refund(sms_body):
    # search for 'http' and then somewhere the word refund and then somewhere '.com'
    match = re.search(r'http.*[rR][eE][fF][uU][nN][dD].*\.com', sms_body)
    if match:
        return [(str(match.group()),)]
    else:
        return []

def find_link_tax(sms_body):
    # search for 'http' and then somewhere the word tax and then somewhere '.com'
    match = re.search(r'http.*[tT][aA][xX].*\.com', sms_body)
    if match:
        return [(str(match.group()),)]
    else:
        return []
    
e.set_ie_function(IEFunction(name='find_link_refund',func=find_link_refund,in_schema=[str],out_schema=[str]))
e.set_ie_function(IEFunction(name='find_link_tax',func=find_link_tax,in_schema=[str],out_schema=[str]))

find_link_refund_rule = Rule(
    head=Relation(name='sms_tax_link', terms=[FreeVar(name='sms_id'), FreeVar(name='sender_id'), FreeVar(name='is_sender_in_contacts'), FreeVar(name='sms_body'),
                                                FreeVar(name='sms_date'), FreeVar(name='is_spam')]),
    body=[
        Relation(name='sms_rel', terms=[FreeVar(name='sms_id'), FreeVar(name='sender_id'), FreeVar(name='is_sender_in_contacts'), FreeVar(name='sms_body'), 
                                        FreeVar(name='sms_date'), FreeVar(name='is_spam')]),
        IERelation(name='find_link_refund', in_terms=[FreeVar(name='sms_body')], out_terms=[FreeVar(name='regex_result')]),
    ])

find_link_tax_rule = Rule(
    head=Relation(name='sms_tax_link', terms=[FreeVar(name='sms_id'), FreeVar(name='sender_id'), FreeVar(name='is_sender_in_contacts'), FreeVar(name='sms_body'),
                                                FreeVar(name='sms_date'), FreeVar(name='is_spam')]),
    body=[
        Relation(name='sms_rel', terms=[FreeVar(name='sms_id'), FreeVar(name='sender_id'), FreeVar(name='is_sender_in_contacts'), FreeVar(name='sms_body'), 
                                        FreeVar(name='sms_date'), FreeVar(name='is_spam')]),
        IERelation(name='find_link_tax', in_terms=[FreeVar(name='sms_body')], out_terms=[FreeVar(name='regex_result')]),
    ])

e.add_rule(find_link_refund_rule,RelationDefinition(name='sms_tax_link', scheme=[str,int,bool,str,str,bool]))
e.add_rule(find_link_tax_rule,RelationDefinition(name='sms_tax_link', scheme=[str,int,bool,str,str,bool]))

query_graph, root_node = e.plan_query(Relation(name='sms_tax_link',terms=[FreeVar(name='sms_id'), FreeVar(name='sender_id'), FreeVar(name='is_sender_in_contacts'), FreeVar(name='sms_body'),
                                                FreeVar(name='sms_date'), FreeVar(name='is_spam')]))
res, profile_data = compute_node(query_graph, root_node)
draw(query_graph)

profile_df = pd.DataFrame(profile_data).T
print(profile_df)
res = res.sort_values(by='sms_id')
res


               count  total_time
get_rel     1.000000    0.000004
rename      7.000000    0.001728
project     9.000000    0.017957
ie_map      2.000000    0.073041
join        2.000000    0.008427
union       1.000000    0.005362
total_time  0.108639    0.108639


,sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam
49,S012345A671112,033-0129472,False,Get your maximum tax refund guaranteed! Click ...,"04/01/2024, 14:45",True
0,S012345A671113,054-5172640,False,See how much you are owed in a tax refund! Sta...,"18/01/2024, 11:45",True
8,S012345A671915,EasyTaxRefund,False,Your refund eligibility expires soon! Verify h...,"15/02/2024, 15:15",True
17,S012345A678909,033-030482,False,"Stav Aviram - Army, Your details have been rec...","22/08/2024, 14:09",True
51,S123456A711123,TaxEasy,False,"Stav Compi, Your refund could be worth up to 2...","19/01/2024, 13:50",True
48,S123456A787654,TaxEasy,False,"Hello Stav Combinatorics Partner, verify your ...","09/01/2024, 15:35",True
12,S123456A788432,TaxOffice15,False,"Due to the situation, you are eligible for a t...","16/01/2024, 20:03",True
32,S123456A789015,EasyTaxRefund,False,Your refund eligibility for 2019 expires soon!...,"23/01/2024, 12:40",True
11,S123456A789016,033-0129472,False,Free tax refund eligibility check available no...,"06/02/2024, 13:50",True
28,S123456A789017,FastRefunds,False,Your eligibility for a refund is pending. Clic...,"02/02/2024, 12:15",True


# Query III - money amount

In [1947]:
#e = Engine()
#e.set_relation(RelationDefinition(name='sms_rel', scheme=[str, int, bool, str, str, bool])) # sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam
#e.add_facts('sms_rel',sms_rel)

def find_money_amount(sms_body):
    # search for a number between 1000 and 100000, can be written with or without a comma
    match = re.search(r'\d{1,3},\d{3}', sms_body)
    if match:
        return [(str(match.group()),)]
    else:
        return []
    
def find_money_currency(sms_body):
    # search for the nis currency
    match = re.search(r'([nN][iI][sS])|₪', sms_body)
    if match:
        return [(str(match.group()),)]
    else:
        return []
  
def find_thousands(sms_body):
    # search for a number between 1k and 100k
    match = re.search(r'\d{1,3}[kK]', sms_body)
    if match:
        return [(str(match.group()),)]
    else:
        return []

e.set_ie_function(IEFunction(name='find_money_amount',func=find_money_amount,in_schema=[str],out_schema=[str]))
e.set_ie_function(IEFunction(name='find_money_currency',func=find_money_currency,in_schema=[str],out_schema=[str]))
e.set_ie_function(IEFunction(name='find_thousands',func=find_thousands,in_schema=[str],out_schema=[str]))

find_money_amount_rule = Rule(
    head=Relation(name='sms_money', terms=[FreeVar(name='sms_id'), FreeVar(name='sender_id'), FreeVar(name='is_sender_in_contacts'), FreeVar(name='sms_body'), 
                                            FreeVar(name='sms_date'), FreeVar(name='is_spam')]),
    body=[
        Relation(name='sms_rel', terms=[FreeVar(name='sms_id'), FreeVar(name='sender_id'), FreeVar(name='is_sender_in_contacts'), FreeVar(name='sms_body'), 
                                        FreeVar(name='sms_date'), FreeVar(name='is_spam')]),
        IERelation(name='find_money_amount', in_terms=[FreeVar(name='sms_body')], out_terms=[FreeVar(name='regex_result')]),
    ])

find_money_currency_rule = Rule(
    head=Relation(name='sms_money', terms=[FreeVar(name='sms_id'), FreeVar(name='sender_id'), FreeVar(name='is_sender_in_contacts'), FreeVar(name='sms_body'), 
                                            FreeVar(name='sms_date'), FreeVar(name='is_spam')]),
    body=[
        Relation(name='sms_rel', terms=[FreeVar(name='sms_id'), FreeVar(name='sender_id'), FreeVar(name='is_sender_in_contacts'), FreeVar(name='sms_body'), 
                                        FreeVar(name='sms_date'), FreeVar(name='is_spam')]),
        IERelation(name='find_money_currency', in_terms=[FreeVar(name='sms_body')], out_terms=[FreeVar(name='regex_result')]),
    ])

find_thousands_rule = Rule(
    head=Relation(name='sms_money', terms=[FreeVar(name='sms_id'), FreeVar(name='sender_id'), FreeVar(name='is_sender_in_contacts'), FreeVar(name='sms_body'), 
                                            FreeVar(name='sms_date'), FreeVar(name='is_spam')]),
    body=[
        Relation(name='sms_rel', terms=[FreeVar(name='sms_id'), FreeVar(name='sender_id'), FreeVar(name='is_sender_in_contacts'), FreeVar(name='sms_body'), 
                                        FreeVar(name='sms_date'), FreeVar(name='is_spam')]),
        IERelation(name='find_thousands', in_terms=[FreeVar(name='sms_body')], out_terms=[FreeVar(name='regex_result')]),
    ])

e.add_rule(find_money_amount_rule,RelationDefinition(name='sms_money', scheme=[str,int,bool,str,str,bool]))
e.add_rule(find_money_currency_rule,RelationDefinition(name='sms_money', scheme=[str,int,bool,str,str,bool]))
e.add_rule(find_thousands_rule,RelationDefinition(name='sms_money', scheme=[str,int,bool,str,str,bool]))

#g = e._inline_db_and_ies_in_graph(e.term_graph)
#print(e.rules_to_ids)
#draw(g)

query_graph, root_node = e.plan_query(Relation(name='sms_money',terms=[FreeVar(name='sms_id'), FreeVar(name='sender_id'), FreeVar(name='is_sender_in_contacts'), FreeVar(name='sms_body'),
                                            FreeVar(name='sms_date'), FreeVar(name='is_spam')]))
res, profile_data = compute_node(query_graph, root_node)
draw(query_graph)

profile_df = pd.DataFrame(profile_data).T
print(profile_df)
res = res.sort_values(by='sms_id')
res

                count  total_time
get_rel      1.000000    0.000003
rename      10.000000    0.001849
project     13.000000    0.012068
ie_map       3.000000    0.085533
join         3.000000    0.006305
union        1.000000    0.002736
total_time   0.109747    0.109747


,sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam
35,S012345A678911,050-8709086,False,"Immediate loan approval for up to 20,000 NIS! ...","21/06/2023, 10:14",False
25,S012345A678914,TaxEasy,False,"Stav Compi Rep, verify your eligibility for a ...","01/02/2024, 10:50",True
67,S123456A711123,TaxEasy,False,"Stav Compi, Your refund could be worth up to 2...","19/01/2024, 13:50",True
31,S123456A789009,TaxMax,False,"Stav Compi, Your 2018 tax refund of 10,400 NIS...","02/04/2024, 21:53",True
12,S123456A789011,033-030571,False,"Hi Nadav army, The year 2024 has arrived, and ...","13/06/2024, 12:02",True
...,...,...,...,...,...,...
30,S936463A791146,Bit,False,Boaz sent you money via bit! 100 NIS are waiti...,"02/04/2024, 20:14",False
13,S936463A791147,Guy My Love,True,Stavi we have to try this https://www.youtube....,"27/01/2022, 09:09",False
64,S970604A859790,Bit,False,Yael Fink sent you money via bit! 45 NIS are w...,"16/04/2024, 10:28",False
8,S982123A547812,CoffeeShopPro,False,Hi Nadav! Your favorite CoffeeShop is offering...,"29/11/2024, 11:45",False


# Query IV - wrong receiver

In [1948]:
#e = Engine()
#e.set_relation(RelationDefinition(name='sms_rel', scheme=[str, int, bool, str, str, bool])) # sms_id, sender_id, is_sender_in_contacts, sms_body, sms_date, is_spam
#e.add_facts('sms_rel',sms_rel)

truecaller_receiver_names = [
        'Stav My Love',
        'Stav CS Technion',
        'Stav Combinatorics Partner',
        'Stav Aviram - Army',
        'Stav Compi',
        'Stav Compi Rep'
        ]

def get_hi_line(sms_body):
    pattern = r"(?i)\bhi\s(" + "|".join(re.escape(name) for name in truecaller_receiver_names) + r")\b.*?[.!?\n]"
    match = re.search(pattern, sms_body)
    if match:
        return [match.group(),]
    else:
        return []
    
def get_hello_line(sms_body):
    pattern = r"(?i)\bhello\s(" + "|".join(re.escape(name) for name in truecaller_receiver_names) + r")\b.*?[.!?\n]"
    match = re.search(pattern, sms_body)
    if match:
        return [match.group(),]
    else:
        return []
    
def get_first_line(sms_body):
    # find the first line of the sms - line ends with \n or . or ? or ! - only if it contains any of the names in truecaller_receiver_names
    pattern = r"(?i)(?:" + "|".join(re.escape(name) for name in truecaller_receiver_names) + r").*?[.!?\n]"
    match = re.search(pattern, sms_body)
    if match:
        return [match.group(),]
    else:
        return []


e.set_ie_function(IEFunction(name='get_hi_line', func=get_hi_line, in_schema=[str], out_schema=[str]))
e.set_ie_function(IEFunction(name='get_hello_line', func=get_hello_line, in_schema=[str], out_schema=[str]))
e.set_ie_function(IEFunction(name='get_first_line', func=get_first_line, in_schema=[str], out_schema=[str]))


hi_rule = Rule(
    head=Relation(name='sms_wrong_receiver', terms=[FreeVar(name='sms_id'), FreeVar(name='sender_id'), FreeVar(name='is_sender_in_contacts'), FreeVar(name='sms_body'), 
                                            FreeVar(name='sms_date'), FreeVar(name='is_spam')]),
    body=[
        Relation(name='sms_rel', terms=[FreeVar(name='sms_id'), FreeVar(name='sender_id'), FreeVar(name='is_sender_in_contacts'), FreeVar(name='sms_body'), 
                                        FreeVar(name='sms_date'), FreeVar(name='is_spam')]),
        IERelation(name='get_hi_line', in_terms=[FreeVar(name='sms_body')], out_terms=[FreeVar(name='regex_result')]),
    ])

hello_rule = Rule(
    head=Relation(name='sms_wrong_receiver', terms=[FreeVar(name='sms_id'), FreeVar(name='sender_id'), FreeVar(name='is_sender_in_contacts'), FreeVar(name='sms_body'),
                                            FreeVar(name='sms_date'), FreeVar(name='is_spam')]),
    body=[
        Relation(name='sms_rel', terms=[FreeVar(name='sms_id'), FreeVar(name='sender_id'), FreeVar(name='is_sender_in_contacts'), FreeVar(name='sms_body'), 
                                        FreeVar(name='sms_date'), FreeVar(name='is_spam')]),
        IERelation(name='get_hello_line', in_terms=[FreeVar(name='sms_body')], out_terms=[FreeVar(name='regex_result')]),
    ])

first_line_rule = Rule(
    head=Relation(name='sms_wrong_receiver', terms=[FreeVar(name='sms_id'), FreeVar(name='sender_id'), FreeVar(name='is_sender_in_contacts'), FreeVar(name='sms_body'),
                                            FreeVar(name='sms_date'), FreeVar(name='is_spam')]),
    body=[
        Relation(name='sms_rel', terms=[FreeVar(name='sms_id'), FreeVar(name='sender_id'), FreeVar(name='is_sender_in_contacts'), FreeVar(name='sms_body'),
                                        FreeVar(name='sms_date'), FreeVar(name='is_spam')]),
        IERelation(name='get_first_line', in_terms=[FreeVar(name='sms_body')], out_terms=[FreeVar(name='regex_result')]),
    ])

e.add_rule(hi_rule,RelationDefinition(name='sms_wrong_receiver', scheme=[str,int,bool,str,str,bool]))
e.add_rule(hello_rule,RelationDefinition(name='sms_wrong_receiver', scheme=[str,int,bool,str,str,bool]))
e.add_rule(first_line_rule,RelationDefinition(name='sms_wrong_receiver', scheme=[str,int,bool,str,str,bool]))

query_graph, root_node = e.plan_query(Relation(name='sms_wrong_receiver',terms=[FreeVar(name='sms_id'), FreeVar(name='sender_id'), FreeVar(name='is_sender_in_contacts'), FreeVar(name='sms_body'),
                                            FreeVar(name='sms_date'), FreeVar(name='is_spam')]))
res, profile_data = compute_node(query_graph, root_node)
draw(query_graph)

profile_df = pd.DataFrame(profile_data).T
print(profile_df)
res = res.sort_values(by='sms_id')
res


                count  total_time
get_rel      1.000000    0.000003
rename      10.000000    0.002156
project     13.000000    0.009118
ie_map       3.000000    0.097894
join         3.000000    0.007102
union        1.000000    0.002069
total_time   0.119374    0.119374


,sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam
20,S012345A678909,033-030482,False,"Stav Aviram - Army, Your details have been rec...","22/08/2024, 14:09",True
30,S012345A678910,033-140125,False,"Stav My Love, Taking a loan due to your situat...","29/10/2023, 15:13",True
27,S012345A678914,TaxEasy,False,"Stav Compi Rep, verify your eligibility for a ...","01/02/2024, 10:50",True
34,S123456A711123,TaxEasy,False,"Stav Compi, Your refund could be worth up to 2...","19/01/2024, 13:50",True
32,S123456A787654,TaxEasy,False,"Hello Stav Combinatorics Partner, verify your ...","09/01/2024, 15:35",True
36,S123456A789009,TaxMax,False,"Stav Compi, Your 2018 tax refund of 10,400 NIS...","02/04/2024, 21:53",True
16,S123456A789012,TaxMax,False,"Stav Compi, Your 2018 tax refund of 10,400 NIS...","02/04/2024, 20:13",True
14,S345678A901232,TaxReturn,False,"Stav Aviram - Army, Regarding the property you...","28/10/2023, 17:33",True
8,S345678A901237,072-3491746,False,"Stav Aviram - Army, good news, tax refunds sim...","25/01/2024, 16:25",True
37,S352109A847623,FastMass,False,"Stav CS Technion, 4 out of 5 people in Israel ...","06/11/2024, 09:27",True


# Query V - suspicious sender

In [1949]:

def sender_digits_only(sender_id, is_sender_in_contacts):
    if is_sender_in_contacts == False:
        match = re.match(r'\d{3}-\d+', sender_id)
        if match:
            return [(match.group(),)]
    return []
    
def is_suspicious_sender_tax(sms_body, is_sender_in_contacts):
    if is_sender_in_contacts == False:
        match = re.search(r'[tT][aA][xX]', sms_body)
        if match:
            return [(True,)]
    return []

def is_suspicious_sender_refund(sms_body, is_sender_in_contacts):
    if is_sender_in_contacts == False:
        match = re.search(r'[rR][eE][fF][uU][nN][dD]', sms_body)
        if match:
            return [(True,)]
    return []
    
e.set_ie_function(IEFunction(name='sender_digits_only', func=sender_digits_only, in_schema=[str, bool], out_schema=[str]))
e.set_ie_function(IEFunction(name='is_suspicious_sender_tax', func=is_suspicious_sender_tax, in_schema=[str, bool], out_schema=[bool]))
e.set_ie_function(IEFunction(name='is_suspicious_sender_refund', func=is_suspicious_sender_refund, in_schema=[str, bool], out_schema=[bool]))

sender_digits_only_rule = Rule(
    head=Relation(name='sms_suspicious_sender', terms=[FreeVar(name='sms_id'), FreeVar(name='sender_id'), FreeVar(name='is_sender_in_contacts'), FreeVar(name='sms_body'),
                                            FreeVar(name='sms_date'), FreeVar(name='is_spam')]),
    body=[
        Relation(name='sms_rel', terms=[FreeVar(name='sms_id'), FreeVar(name='sender_id'), FreeVar(name='is_sender_in_contacts'), FreeVar(name='sms_body'), 
                                        FreeVar(name='sms_date'), FreeVar(name='is_spam')]),
        IERelation(name='sender_digits_only', in_terms=[FreeVar(name='sender_id'), FreeVar(name='is_sender_in_contacts')], out_terms=[FreeVar(name='regex_result')]),
    ])

is_suspicious_sender_tax_rule = Rule(
    head=Relation(name='sms_suspicious_sender', terms=[FreeVar(name='sms_id'), FreeVar(name='sender_id'), FreeVar(name='is_sender_in_contacts'), FreeVar(name='sms_body'), 
                                            FreeVar(name='sms_date'), FreeVar(name='is_spam')]),
    body=[
        Relation(name='sms_rel', terms=[FreeVar(name='sms_id'), FreeVar(name='sender_id'), FreeVar(name='is_sender_in_contacts'), FreeVar(name='sms_body'),
                                        FreeVar(name='sms_date'), FreeVar(name='is_spam')]),
        IERelation(name='is_suspicious_sender_tax', in_terms=[FreeVar(name='sms_body'), FreeVar(name='is_sender_in_contacts')], out_terms=[FreeVar(name='regex_result')]),
    ])

is_suspicious_sender_refund_rule = Rule(
    head=Relation(name='sms_suspicious_sender', terms=[FreeVar(name='sms_id'), FreeVar(name='sender_id'), FreeVar(name='is_sender_in_contacts'), FreeVar(name='sms_body'), 
                                            FreeVar(name='sms_date'), FreeVar(name='is_spam')]),
    body=[
        Relation(name='sms_rel', terms=[FreeVar(name='sms_id'), FreeVar(name='sender_id'), FreeVar(name='is_sender_in_contacts'), FreeVar(name='sms_body'), 
                                        FreeVar(name='sms_date'), FreeVar(name='is_spam')]),
        IERelation(name='is_suspicious_sender_refund', in_terms=[FreeVar(name='sms_body'), FreeVar(name='is_sender_in_contacts')], out_terms=[FreeVar(name='regex_result')]),
    ])

e.add_rule(sender_digits_only_rule,RelationDefinition(name='sms_suspicious_sender', scheme=[str,int,bool,str,str,bool,bool]))
e.add_rule(is_suspicious_sender_tax_rule,RelationDefinition(name='sms_suspicious_sender', scheme=[str,int,bool,str,str,bool,bool]))
e.add_rule(is_suspicious_sender_refund_rule,RelationDefinition(name='sms_suspicious_sender', scheme=[str,int,bool,str,str,bool,bool]))

query_graph, root_node = e.plan_query(Relation(name='sms_suspicious_sender',terms=[FreeVar(name='sms_id'), FreeVar(name='sender_id'), FreeVar(name='is_sender_in_contacts'), FreeVar(name='sms_body'),
                                            FreeVar(name='sms_date'), FreeVar(name='is_spam')]))
res, profile_data = compute_node(query_graph, root_node)
draw(query_graph)

profile_df = pd.DataFrame(profile_data).T
print(profile_df)
res

                count  total_time
get_rel      1.000000    0.000003
rename      10.000000    0.001565
project     13.000000    0.009397
ie_map       3.000000    0.037335
join         3.000000    0.006927
union        1.000000    0.003376
total_time   0.059861    0.059861


,sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam
0,S352109A847629,TaxService,False,Maximize your tax refund in minutes! Click her...,"03/02/2024, 14:20",True
1,S123456A789014,054-6123656,False,Mortgage benefits now available! Up to 85 home...,"24/07/2023, 18:27",False
2,S456789A012349,SmartRefund,False,Important! Check your tax refund eligibility t...,"09/02/2024, 12:50",True
3,S476521A098768,TaxService,False,Tax refunds made easy! Check your eligibility ...,"21/01/2024, 09:25",True
4,S789012A345676,Missim,False,You might also receive such a message! Hello S...,"06/08/2024, 15:52",True
...,...,...,...,...,...,...
102,S567890A123456,TAXES,False,"Stav Combinatorics Partner, reminder for you t...","05/03/2023, 20:42",True
103,S352109A847624,TaxOffice15,False,"Due to the situation, you are eligible for a t...","06/03/2024, 20:12",True
104,S123456A789013,RefundTax,False,Check if you are eligible for a tax refund for...,"29/11/2021, 18:12",True
105,S345678A901236,TaxService,False,Maximize your tax refund in minutes! Click her...,"11/01/2024, 09:00",True


# Putting it together

In [1950]:
full_tax_year_rule = Rule(
    head=Relation(name='sms_tax', terms=[FreeVar(name='sms_id'), FreeVar(name='sender_id'), FreeVar(name='is_sender_in_contacts'), FreeVar(name='sms_body'), 
                                            FreeVar(name='sms_date'), FreeVar(name='is_spam')]),
    body=[Relation(name='sms_tax_year', terms=[FreeVar(name='sms_id'), FreeVar(name='sender_id'), FreeVar(name='is_sender_in_contacts'), FreeVar(name='sms_body'), 
                                            FreeVar(name='sms_date'), FreeVar(name='is_spam')])],
    )

full_tax_link_rule = Rule(
    head=Relation(name='sms_tax', terms=[FreeVar(name='sms_id'), FreeVar(name='sender_id'), FreeVar(name='is_sender_in_contacts'), FreeVar(name='sms_body'), 
                                            FreeVar(name='sms_date'), FreeVar(name='is_spam')]),
    body=[Relation(name='sms_tax_link', terms=[FreeVar(name='sms_id'), FreeVar(name='sender_id'), FreeVar(name='is_sender_in_contacts'), FreeVar(name='sms_body'),
                                            FreeVar(name='sms_date'), FreeVar(name='is_spam')])],
    )

full_tax_money_rule = Rule(
    head=Relation(name='sms_tax', terms=[FreeVar(name='sms_id'), FreeVar(name='sender_id'), FreeVar(name='is_sender_in_contacts'), FreeVar(name='sms_body'), 
                                            FreeVar(name='sms_date'), FreeVar(name='is_spam')]),
    body=[Relation(name='sms_money', terms=[FreeVar(name='sms_id'), FreeVar(name='sender_id'), FreeVar(name='is_sender_in_contacts'), FreeVar(name='sms_body'), 
                                            FreeVar(name='sms_date'), FreeVar(name='is_spam')])],
    )

full_tax_wrong_receiver_rule = Rule(
    head=Relation(name='sms_tax', terms=[FreeVar(name='sms_id'), FreeVar(name='sender_id'), FreeVar(name='is_sender_in_contacts'), FreeVar(name='sms_body'), 
                                            FreeVar(name='sms_date'), FreeVar(name='is_spam')]),
    body=[Relation(name='sms_wrong_receiver', terms=[FreeVar(name='sms_id'), FreeVar(name='sender_id'), FreeVar(name='is_sender_in_contacts'), FreeVar(name='sms_body'), 
                                            FreeVar(name='sms_date'), FreeVar(name='is_spam')])],
    )

full_tax_suspicous_sender_rule = Rule(
    head=Relation(name='sms_tax', terms=[FreeVar(name='sms_id'), FreeVar(name='sender_id'), FreeVar(name='is_sender_in_contacts'), FreeVar(name='sms_body'), 
                                            FreeVar(name='sms_date'), FreeVar(name='is_spam')]),
    body=[Relation(name='sms_suspicious_sender', terms=[FreeVar(name='sms_id'), FreeVar(name='sender_id'), FreeVar(name='is_sender_in_contacts'), FreeVar(name='sms_body'), 
                                            FreeVar(name='sms_date'), FreeVar(name='is_spam')])],
    )


e.add_rule(full_tax_year_rule,RelationDefinition(name='sms_tax', scheme=[str,int,bool,str,str,bool]))
e.add_rule(full_tax_link_rule,RelationDefinition(name='sms_tax', scheme=[str,int,bool,str,str,bool]))
e.add_rule(full_tax_money_rule,RelationDefinition(name='sms_tax', scheme=[str,int,bool,str,str,bool]))
e.add_rule(full_tax_wrong_receiver_rule,RelationDefinition(name='sms_tax', scheme=[str,int,bool,str,str,bool]))
e.add_rule(full_tax_suspicous_sender_rule,RelationDefinition(name='sms_tax', scheme=[str,int,bool,str,str,bool]))

#g = e._inline_db_and_ies_in_graph(e.term_graph)
#print(e.rules_to_ids)
#draw(g)

query_graph, root_node = e.plan_query(Relation(name='sms_tax',terms=[FreeVar(name='sms_id'), FreeVar(name='sender_id'), FreeVar(name='is_sender_in_contacts'), FreeVar(name='sms_body'),
                                            FreeVar(name='sms_date'), FreeVar(name='is_spam')]))
res, profile_data = compute_node(query_graph, root_node)
draw(query_graph)

profile_df = pd.DataFrame(profile_data).T
print(profile_df)
res = res.sort_values(by='sms_id')
res


                count  total_time
get_rel      1.000000    0.000003
rename      45.000000    0.006271
project     63.000000    0.026511
ie_map      13.000000    0.222753
join        13.000000    0.021053
union        6.000000    0.012211
total_time   0.292608    0.292608


,sms_id,sender_id,is_sender_in_contacts,sms_body,sms_date,is_spam
134,S012345A671112,033-0129472,False,Get your maximum tax refund guaranteed! Click ...,"04/01/2024, 14:45",True
74,S012345A671113,054-5172640,False,See how much you are owed in a tax refund! Sta...,"18/01/2024, 11:45",True
83,S012345A671915,EasyTaxRefund,False,Your refund eligibility expires soon! Verify h...,"15/02/2024, 15:15",True
20,S012345A678909,033-030482,False,"Stav Aviram - Army, Your details have been rec...","22/08/2024, 14:09",True
67,S012345A678910,033-140125,False,"Stav My Love, Taking a loan due to your situat...","29/10/2023, 15:13",True
...,...,...,...,...,...,...
100,S936463A791146,Bit,False,Boaz sent you money via bit! 100 NIS are waiti...,"02/04/2024, 20:14",False
15,S936463A791147,Guy My Love,True,Stavi we have to try this https://www.youtube....,"27/01/2022, 09:09",False
66,S970604A859790,Bit,False,Yael Fink sent you money via bit! 45 NIS are w...,"16/04/2024, 10:28",False
82,S982123A547812,CoffeeShopPro,False,Hi Nadav! Your favorite CoffeeShop is offering...,"29/11/2024, 11:45",False


# Success Rate

In [1951]:
success_rate_engine = Engine()
# add previous res to be the success_rate_engine base DB
success_rate_engine.set_relation(RelationDefinition(name='sms_tax', scheme=[str, int, bool, str, str, bool, str, str, str]))
success_rate_engine.add_facts('sms_tax', 
                              res)

success_rate_engine.set_agg_function(AGGFunction(name='count',func='count',in_schema=[str],out_schema=[int]))

# Rule to count the number of rows with each tag
count_total_rule = Rule(
    head=Relation(name='tag_total_count', terms=[FreeVar(name='tag'), FreeVar(name='tag')], agg=[None, 'count']),
    body=[Relation(name='sms_tax', terms=[FreeVar(name='sms_id'), FreeVar(name='sender_id'), FreeVar(name='is_sender_in_contacts'), FreeVar(name='sms_body'), 
                                          FreeVar(name='sms_date'), FreeVar(name='is_spam'), FreeVar(name='tag'), FreeVar(name='found_by_regex'), FreeVar(name='regex_result')])],
)

# Rule to count the number of rows with each tag where is_spam=True
count_spam_rule = Rule(
    head=Relation(name='tag_spam_count', terms=[FreeVar(name='tag'), FreeVar(name='is_spam')], agg=[None, 'count']),
    body=[Relation(name='sms_tax', terms=[FreeVar(name='sms_id'), FreeVar(name='sender_id'), FreeVar(name='is_sender_in_contacts'), FreeVar(name='sms_body'), 
                                          FreeVar(name='sms_date'), FreeVar(name='is_spam'), FreeVar(name='tag'), FreeVar(name='found_by_regex'), FreeVar(name='regex_result')]),
          Relation(name='sms_tax', terms=[FreeVar(name='sms_id'), FreeVar(name='sender_id'), FreeVar(name='is_sender_in_contacts'), FreeVar(name='sms_body'), 
                                          FreeVar(name='sms_date'), True, FreeVar(name='tag'), FreeVar(name='found_by_regex'), FreeVar(name='regex_result')])],
)

# Rule to calculate the success rate
def calculate_success_rate(spam_count, total_count):
    if total_count > 0:
        return [(spam_count / total_count,)]
    return [(0,)]

success_rate_engine.set_ie_function(IEFunction(name='calculate_success_rate', func=calculate_success_rate, in_schema=[int, int], out_schema=[float]))

success_rate_rule = Rule(
    head=Relation(name='tag_success_rate', terms=[FreeVar(name='tag'), FreeVar(name='success_rate')]),
    body=[Relation(name='tag_spam_count', terms=[FreeVar(name='tag'), FreeVar(name='spam_count')]),
          Relation(name='tag_total_count', terms=[FreeVar(name='tag'), FreeVar(name='total_count')]),
          IERelation(name='calculate_success_rate', in_terms=[FreeVar(name='spam_count'), FreeVar(name='total_count')], out_terms=[FreeVar(name='success_rate')])],
)

# Add the rules to the engine
success_rate_engine.add_rule(count_total_rule, RelationDefinition(name='tag_total_count', scheme=[str, int]))
success_rate_engine.add_rule(count_spam_rule, RelationDefinition(name='tag_spam_count', scheme=[str, int]))
success_rate_engine.add_rule(success_rate_rule, RelationDefinition(name='tag_success_rate', scheme=[str, float]))

# Run the query to get the success rate for each tag
query_graph, root_node = success_rate_engine.plan_query(Relation(name='tag_success_rate',terms=[FreeVar(name='tag'), FreeVar(name='success_rate')]))
draw(query_graph)
res, _ = compute_node(query_graph, root_node)
res

Exception: During excution of node 11 with args [                  0            1      2  \
0    S352109A847629   TaxService  False   
2    S456789A012349  SmartRefund  False   
4    S476521A098768   TaxService  False   
5    S789012A345676       Missim  False   
6    S345678A901234         CTAX  False   
..              ...          ...    ...   
136  S567890A123456        TAXES  False   
137  S352109A847624  TaxOffice15  False   
138  S123456A789013    RefundTax  False   
139  S345678A901236   TaxService  False   
140  S352109A847623     FastMass  False   

                                                     3                  4  \
0    Maximize your tax refund in minutes! Click her...  03/02/2024, 14:20   
2    Important! Check your tax refund eligibility t...  09/02/2024, 12:50   
4    Tax refunds made easy! Check your eligibility ...  21/01/2024, 09:25   
5    You might also receive such a message! Hello S...  06/08/2024, 15:52   
6    Check your 2017 tax refund eligibility now! 